In [2]:
import numpy as np
import pandas as pd

import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

In [3]:
conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

cursor.execute('''SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';''')

[row[0] for row in cursor.fetchall()]


['ads_raw', 'ads_cleaned']

In [ ]:
# Grant Supabase roles access to ads_cleaned so it appears in the Table Editor UI
cursor.execute("""
    GRANT ALL ON public.ads_cleaned TO anon, authenticated, service_role;
""")
conn.commit()
print("Permissions granted. Hard refresh the Supabase UI (Ctrl+Shift+R).")


In [12]:
cursor.execute("DROP TABLE IF EXISTS ads_cleaned_backup")
conn.commit()

In [5]:
query = '''SELECT * FROM ads_raw '''
cursor.execute(query)
rows = cursor.fetchall()

columns = [description[0] for description in cursor.description]

ads_raw = pd.DataFrame(data=rows, columns=columns)

In [6]:
ads_raw.head()

,hash_id,title,link,img_url,total_price_eur,price_m2_eur,price_m2_bgn,size_m2,description,floor,akt16,energy_class,potreblenie,broker_commision,additional_notes,status,extras,scraped_at,last_updated
0,191cff973bff3518de40c8019f6c3d5fea7174c95837aa...,"Имот - продава Двустаен апартамент, в София, М...",https://www.imoti.net/bg/obiava/prodava/sofia/...,https://www.imoti.net/web/files/obiavi/6256545...,None,2 263,4 426.32,57,"Двустаен апартамент с площ 57,01м2 /ЗП 48,86м2...",4 от 4,Да,N/A,N/A,Да,Посочената цена не включва местни данъци и такси.,done,NaN,2026-06-07 12:23:33.916334,2026-06-07 13:04:28.861833
1,9eb41908d4e61b7008f29eb1688015f63ff3cdf229084b...,"Имот - продава Двустаен апартамент, в София, М...",https://www.imoti.net/bg/obiava/prodava/sofia/...,https://www.imoti.net/web/files/obiavi/6256546...,None,2 263,4 426.32,57,"Двустаен апартамент с площ 57,01м2 /ЗП 48,86м2...",4 от 4,Да,N/A,N/A,Да,Посочената цена не включва местни данъци и такси.,done,NaN,2026-06-07 12:23:34.038430,2026-06-07 13:04:28.984031
2,dbc5d9466f8d880697f9fee292fe7c5fc84663ec10bcd4...,"Имот - продава Двустаен апартамент, в София, В...",https://www.imoti.net/bg/obiava/prodava/sofia/...,https://www.imoti.net/web/files/obiavi/6262276...,None,2 580,5 046.04,50,Компания `ЕКС` представя на Вашето внимание фу...,5 от 8,Да,N/A,N/A,Да,Посочената цена не включва местни данъци и такси.,done,Обзаведен; ТЕЦ; Асансьор,2026-06-07 12:23:34.158548,2026-06-07 13:04:29.106281
3,6833ef4bed6c98041e1c7433fc60698103a43595c45950...,"Имот - продава Двустаен апартамент, в София, В...",https://www.imoti.net/bg/obiava/prodava/sofia/...,https://www.imoti.net/web/files/obiavi/6246165...,None,2 804,5 484.83,46,Реф. 013417 Агенция за недвижими имоти BAS P...,5 от 8,Да,N/A,N/A,Да,Посочената цена не включва местни данъци и такси.,done,Обзаведен,2026-06-07 12:23:34.277707,2026-06-07 13:04:29.228334
4,65707b6e4a05c4b88ad12e0779c1fc35bd125f66133bc6...,"Имот - продава Едностаен апартамент, в София, ...",https://www.imoti.net/bg/obiava/prodava/sofia/...,https://www.imoti.net/web/files/obiavi/6248616...,None,3 167,6 193.46,42,Референтен номер: 2109493 Революшън Естейт пре...,3 от 5,Да,N/A,N/A,Да,Посочената цена не включва местни данъци и такси.,done,NaN,2026-06-07 12:20:29.961803,2026-06-07 12:45:37.180873


In [5]:
ads_cleaned["type_of_estate"].unique().tolist()

['гараж', 'парцел', 'магазин', 'жилище']

In [4]:
ads_cleaned[ads_cleaned["price_m2_eur"] == 146]

,hash_id,title,imgUrl,link,neighbourhood,type_of_estate,total_price_eur,price_m2_eur,price_m2_bgn,size_m2,nr_of_rooms,description,floor,akt16,energy_class,potreblenie,broker_commision,additional_notes,extras
349,a88c6472a6a902b13b5f01f86836201937f1f7af6cc98c...,"Имот - продава Парцел, в София, Банкя (гр.)",https://www.imoti.net/web/files/obiavi/6110812...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Банкя (гр.),парцел,73000.0,146.0,285.55,500.0,NaN,Компания `ЕКС` представя на Вашето внимание УП...,NaN,NaN,NaN,NaN,0.0,Посочената цена не включва местни данъци и такси.,Регулация; Ток; Водопровод; За жилищно строите...
2896,b1d3f9878d6842c3e6877a4fe4b07c9a1379128f2f1522...,"Имот - продава Парцел, в София, Дружба 2",https://www.imoti.net/web/files/obiavi/6244106...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Дружба 2,парцел,190530.0,146.0,284.76,1305.0,NaN,Представяме на вашето внимание ексклузивна офе...,NaN,NaN,NaN,NaN,0.0,Посочената цена не включва местни данъци и такси.,Ток; Равен; За промишлено строителство
2897,d15e6fb353fdcdd645d91e87e7a08922d14376adac777c...,"Имот - продава Парцел, в София, Дружба 2",https://www.imoti.net/web/files/obiavi/6120731...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Дружба 2,парцел,190530.0,146.0,284.76,1305.0,NaN,Представяме на вашето внимание ексклузивна офе...,NaN,NaN,NaN,NaN,0.0,Посочената цена не включва местни данъци и такси.,Ток; Равен; За промишлено строителство
3611,b0e1e306ec380464278372fd51a16ad66f8f04a0927979...,"Имот - продава Парцел, в София",https://www.imoti.net/web/files/obiavi/6075424...,https://www.imoti.net/bg/obiava/prodava--v-str...,NaN,парцел,210678.0,146.0,284.63,1443.0,NaN,Представяме поземлен имот за продажба в урбани...,NaN,NaN,NaN,NaN,1.0,Посочената цена не включва местни данъци и такси.,EMPTY


In [14]:
conn = sqlite3.connect("../scraper/data/ads_storage.db")
cursor = conn.cursor()

query = '''SELECT * FROM ads_cleaned '''
cursor.execute(query)
rows = cursor.fetchall()

columns = [description[0] for description in cursor.description]

ads_cleaned = pd.DataFrame(data=rows, columns=columns)
ads_cleaned.head()


,hash_id,title,imgUrl,link,neighbourhood,type_of_estate,total_price_eur,price_m2_eur,price_m2_bgn,size_m2,nr_of_rooms,description,floor,akt16,energy_class,potreblenie,broker_commision,additional_notes,extras
0,f86956dd334888697e09f28acb103cf8c9dfa00f0bc5d6...,"Имот - продава Гараж, паркомясто, в София, Обо...",https://www.imoti.net/web/files/obiavi/6196041...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Оборище,гараж,39996.0,3333.0,6519.43,12.0,NaN,"Строително-инвестиционна компания Идея Хоум, п...",NaN,1.0,NaN,NaN,0.0,Посочената цена не включва местни данъци и такси.,ТЕЦ
1,54f96cec5e494e296451bd6588e2153de45b96bc8846ab...,"Имот - продава Парцел, в София, Банкя (гр.)",https://www.imoti.net/web/files/obiavi/6174938...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Банкя (гр.),парцел,69888.0,78.0,152.78,896.0,NaN,УПИ с площ от 896 кв. м. в местността Купен до...,NaN,NaN,NaN,NaN,1.0,Посочената цена не включва местни данъци и такси.,Регулация
2,db86942d1b1b8fc970e32022d5f548165d9813d4212ad3...,"Имот - продава Магазин, в София, Гео Милев",https://www.imoti.net/web/files/obiavi/5762781...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Гео Милев,магазин,87885.0,2835.0,5545.72,31.0,NaN,"Dotwon Real Estate, част от Dotwon Group, пред...",0.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,
3,ca420156d8e9fbfbd273afed51b5b294de1496404facea...,"Имот - продава Едностаен апартамент, в София, ...",https://www.imoti.net/web/files/obiavi/5945620...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Овча Купел,жилище,92512.0,1888.0,3692.13,49.0,1.0,ИМА ВЪЗМОЖНОСТ ЗА ЗАКУПУВАНЕ НА ПАРКОМЯСТО. Пр...,1.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,Необзаведен; Паркомясто
4,528685330d09ed04f6d4182b1b75bff4340766ce5aac35...,"Имот - продава Едностаен апартамент, в София, ...",https://www.imoti.net/web/files/obiavi/5883830...,https://www.imoti.net/bg/obiava/prodava/sofia/...,Фондови Жилища,жилище,95975.0,1745.0,3413.81,55.0,1.0,След основен ремонт. Едностаен апартамент в ту...,2.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,


In [9]:
len(ads_cleaned[ads_cleaned["imgUrl"] == "EMPTY"])

6135

In [11]:
ads_cleaned["nr_of_rooms"].unique().tolist()

[nan, 1.0, 2.0, 3.0, 4.0]

In [4]:
conn = sqlite3.connect("../scraper/data/ads_storage.db")
cursor = conn.cursor()

query = '''SELECT hash_id FROM ads_cleaned WHERE nr_of_rooms = 1'''
cursor.execute(query)
rows = cursor.fetchall()

columns = [description[0] for description in cursor.description]

ads_cleaned = pd.DataFrame(data=rows, columns=columns)
ads_cleaned.head()

,hash_id
0,ca420156d8e9fbfbd273afed51b5b294de1496404facea...
1,528685330d09ed04f6d4182b1b75bff4340766ce5aac35...
2,32693a15a3d41f6871dc5feffed1db7a0885aee6fa56fb...
3,61aa28d52d4c0bcc691567726316f400c98147388bd4bc...
4,64d07d8d6c1dec8dc577dc29b99b395b31c2030f9c5a9c...


In [20]:
conn = sqlite3.connect("../scraper/data/ads_storage.db")
cursor = conn.cursor()

query = '''SELECT * FROM ads_cleaned_backup'''
cursor.execute(query)
rows = cursor.fetchall()

columns = [description[0] for description in cursor.description]

ads_cleaned_backup = pd.DataFrame(data=rows, columns=columns)
ads_cleaned_backup.info()

<class 'pandas.DataFrame'>
RangeIndex: 6221 entries, 0 to 6220
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   hash_id           6221 non-null   str    
 1   title             6221 non-null   str    
 2   link              6221 non-null   str    
 3   neighbourhood     6110 non-null   str    
 4   type_of_estate    6221 non-null   str    
 5   price_m2_eur      6130 non-null   float64
 6   price_m2_bgn      6130 non-null   float64
 7   size_m2           6126 non-null   float64
 8   description       6221 non-null   str    
 9   floor             5435 non-null   float64
 10  akt16             5276 non-null   float64
 11  energy_class      5574 non-null   str    
 12  potreblenie       5574 non-null   str    
 13  broker_commision  6106 non-null   float64
 14  additional_notes  6134 non-null   str    
 15  extras            6221 non-null   str    
 16  total_price_eur   6213 non-null   float64
dtypes: flo

In [8]:
ads_cleaned["akt16"].value_counts()

akt16
1.0    4515
0.0     761
Name: count, dtype: int64

In [13]:
ads_cleaned.iloc[1, 1]

'Имот - продава Парцел, в София, Банкя (гр.)'

In [15]:
"Многостаен" in ads_cleaned.iloc[1, 1]

False

In [16]:
# create a 'nr of rooms' column
def extract_number_of_rooms(text):
    if "Едностаен" in text:
        return 1
    elif "Двyстаен" in text:
        return 2
    elif "Тристаен" in text:
        return 3
    elif "Четиристаен" in text or "Многостаен" in text:
        return 4
    else:
        return np.nan
        

ads_cleaned_copy = ads_cleaned.copy()
ads_cleaned_copy["nr_of_rooms"] = ads_cleaned_copy["title"].apply(extract_number_of_rooms)
ads_cleaned_copy


,hash_id,title,link,neighbourhood,type_of_estate,price_m2_eur,price_m2_bgn,size_m2,description,floor,akt16,energy_class,potreblenie,broker_commision,additional_notes,extras,total_price_eur,nr_of_rooms
0,f86956dd334888697e09f28acb103cf8c9dfa00f0bc5d6...,"Имот - продава Гараж, паркомясто, в София, Обо...",https://www.imoti.net/bg/obiava/prodava/sofia/...,Оборище,гараж,3333.0,6519.43,12.0,"Строително-инвестиционна компания Идея Хоум, п...",NaN,1.0,NaN,NaN,0.0,Посочената цена не включва местни данъци и такси.,ТЕЦ,39996.0,NaN
1,54f96cec5e494e296451bd6588e2153de45b96bc8846ab...,"Имот - продава Парцел, в София, Банкя (гр.)",https://www.imoti.net/bg/obiava/prodava/sofia/...,Банкя (гр.),парцел,78.0,152.78,896.0,УПИ с площ от 896 кв. м. в местността Купен до...,NaN,NaN,NaN,NaN,1.0,Посочената цена не включва местни данъци и такси.,Регулация,69888.0,NaN
2,db86942d1b1b8fc970e32022d5f548165d9813d4212ad3...,"Имот - продава Магазин, в София, Гео Милев",https://www.imoti.net/bg/obiava/prodava/sofia/...,Гео Милев,магазин,2835.0,5545.72,31.0,"Dotwon Real Estate, част от Dotwon Group, пред...",0.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,,87885.0,NaN
3,ca420156d8e9fbfbd273afed51b5b294de1496404facea...,"Имот - продава Едностаен апартамент, в София, ...",https://www.imoti.net/bg/obiava/prodava/sofia/...,Овча Купел,жилище,1888.0,3692.13,49.0,ИМА ВЪЗМОЖНОСТ ЗА ЗАКУПУВАНЕ НА ПАРКОМЯСТО. Пр...,1.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,Необзаведен; Паркомясто,92512.0,1.0
4,528685330d09ed04f6d4182b1b75bff4340766ce5aac35...,"Имот - продава Едностаен апартамент, в София, ...",https://www.imoti.net/bg/obiava/prodava/sofia/...,Фондови Жилища,жилище,1745.0,3413.81,55.0,След основен ремонт. Едностаен апартамент в ту...,2.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,,95975.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6216,e7c20c29939aa363004c2b270277574739bf5775761580...,"Имот - продава Тристаен апартамент, в София, М...",https://www.imoti.net/bg/obiava/prodava/sofia/...,Манастирски ливади Изток,жилище,2680.0,5241.01,111.0,"""Бориса Апартментс 2"" е затворен комплекс с мо...",1.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,EMPTY,297480.0,3.0
6217,5e859d392492ad71fb2ecdb32733badcc69b2bb2a84745...,"Имот - продава Тристаен апартамент, в София, Д...",https://www.imoti.net/bg/obiava/prodava/sofia/...,Дружба 2,жилище,2361.0,4617.62,126.0,New Estates има удоволствието да представя про...,3.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,Необзаведен,297486.0,3.0
6218,766bd265e225f4db1bfeda0ba5f8b43fdea9782d92a21c...,"Имот - продава Тристаен апартамент, в София, Д...",https://www.imoti.net/bg/obiava/prodava--v-str...,Дружба 2,жилище,2380.0,4654.56,125.0,ЕРА Младост предлага БЕЗ КОМИСОННА! Двустайни ...,3.0,NaN,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,Необзаведен; Ток; Водопровод; Канализация; ТЕЦ...,297500.0,3.0
6219,d92ab7126b07222b7e5189b66cb79e91d7cad2812a0f8a...,"Имот - продава Тристаен апартамент, в София, П...",https://www.imoti.net/bg/obiava/prodava/sofia/...,Панчарево (с.),жилище,2500.0,4889.58,119.0,ВИЖТЕ ВИДЕО ОТ ЗАСНЕМАНЕ С ДРОН: https://bit.l...,2.0,1.0,N/A,N/A,1.0,Посочената цена не включва местни данъци и такси.,Необзаведен; Гараж; Асансьор; Паркомясто,297500.0,3.0


In [ ]:
def add_nr_of_rroms_column():
    # Add nr_of_rooms column to the database
    try:
        cursor.execute("ALTER TABLE ads_cleaned ADD COLUMN nr_of_rooms SMALLINT")
        conn.commit()
        print("Column added.")
    except Exception as e:
        print(f"Column may already exist: {e}")

    # Update each row with the computed value
    for _, row in ads_cleaned_copy.iterrows():
        cursor.execute(
            "UPDATE ads_cleaned SET nr_of_rooms = ? WHERE hash_id = ?",
            (None if pd.isna(row["nr_of_rooms"]) else int(row["nr_of_rooms"]), row["hash_id"])
        )
    conn.commit()
    print("Done.")


In [8]:
def count_by_title_keyword(df, keyword):
    return df['title'].str.contains(keyword, case=False, na=False).sum()

count_by_title_keyword(ads_cleaned, "Едностаен")


np.int64(279)